# NLP Experiment 7 – Summarization with a pre-trained Transformer (T5) + ROUGE
**Runtime:** Python 3 + **T4 GPU** (Runtime → Change runtime type). Runtime → Run all.

Model: **T5-small**, a pre-trained encoder–decoder Transformer ("text-to-text": input `summarize: <article>`).
(a) **fine-tune** it on CNN/DailyMail news; (b) generate **abstractive** and **extractive** summaries;
(c) evaluate with **ROUGE**. No GPU? Skip the training cell – the pre-trained T5 already summarises.

In [ ]:
!pip install -q transformers datasets rouge_score sentencepiece accelerate

In [ ]:
import re
from collections import Counter
import torch
import pandas as pd
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from rouge_score import rouge_scorer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Load data and the pre-trained model

In [ ]:
data = load_dataset("abisee/cnn_dailymail", "3.0.0")
train_data = data["train"].select(range(2000))     # small subset so training takes a few minutes
test_data = data["test"].select(range(20))

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print(train_data[0]["article"][:300])
print("REFERENCE:", train_data[0]["highlights"])

## (a) Fine-tune T5 on the summarization dataset
Tokenize: article → `input_ids` (max 512 tokens), reference summary → `labels` (max 128 tokens).

In [ ]:
def tokenize(batch):
    inputs = tokenizer(["summarize: " + a for a in batch["article"]], max_length=512, truncation=True)
    labels = tokenizer(text_target=batch["highlights"], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokens = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)

args = Seq2SeqTrainingArguments(
    output_dir="t5-summarizer",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    learning_rate=3e-4,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokens,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)
trainer.train()          # skip this cell if there is no GPU

## (b) Abstractive summary (model writes new sentences)

In [ ]:
def abstractive_summary(article):
    inputs = tokenizer("summarize: " + article, return_tensors="pt", max_length=512, truncation=True).to(device)
    ids = model.generate(**inputs, max_length=100, min_length=30, num_beams=4, early_stopping=True)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

article = test_data[0]["article"]
print("ARTICLE:", article[:600], "...\n")
print("ABSTRACTIVE:", abstractive_summary(article))
print("\nREFERENCE:", test_data[0]["highlights"])

## (b) Extractive summary (pick the most important original sentences)

In [ ]:
def extractive_summary(text, n=3):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    words = re.findall(r"[a-z]+", text.lower())
    freq = Counter(words)

    scores = {}
    for s in sentences:
        s_words = re.findall(r"[a-z]+", s.lower())
        if s_words:
            scores[s] = sum(freq[w] for w in s_words) / len(s_words)    # average word frequency

    best = sorted(scores, key=scores.get, reverse=True)[:n]
    return " ".join(s for s in sentences if s in best)                 # keep original order

print("EXTRACTIVE:", extractive_summary(article))

## (c) Evaluate with ROUGE
ROUGE compares a summary with the human reference: ROUGE-1 = overlapping words, ROUGE-2 = overlapping word pairs,
ROUGE-L = longest common subsequence. F-measure from 0 to 1 (higher = better).

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rows = []
for item in test_data:
    ref = item["highlights"]
    for method, summary in [("abstractive (T5)", abstractive_summary(item["article"])),
                            ("extractive", extractive_summary(item["article"]))]:
        s = scorer.score(ref, summary)
        rows.append({"method": method,
                     "rouge1": s["rouge1"].fmeasure,
                     "rouge2": s["rouge2"].fmeasure,
                     "rougeL": s["rougeL"].fmeasure})

results = pd.DataFrame(rows).groupby("method").mean().round(4)
results

In [ ]:
results.plot(kind="bar", figsize=(8, 4), title="ROUGE: abstractive vs extractive", rot=0)